In [ ]:
# =========================
# MODIFIED FOR SAFE EMAILS – BASED ON YOUR ORIGINAL
# =========================

from google.colab import drive
drive.mount('/content/drive')

# Install once (same as before)
!pip install -q --upgrade datasets pandas

import pandas as pd
from datasets import Dataset
import os

# ───── YOUR PATHS (change only if needed) ─────
DATASET_PATH = "/content/drive/MyDrive/Realsafe.xlsx"
OUTPUT_DIR   = "/content/drive/MyDrive/SafeEmailProject"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ───── Load Excel file exactly as it is ─────
df = pd.read_excel(DATASET_PATH)

df = df[df['label'] == 0].copy()

print(f"Loaded and filtered → {len(df)} safe emails")

# ───── One single common prompt (updated for safe emails) ─────
COMMON_PROMPT = "Generate a realistic legitimate email."

# ───── Convert every row to Vicuna chat format ─────
chat_data_strings = []
for _, row in df.iterrows():
    subject = str(row['subject']) if pd.notna(row['subject']) else "Important Update"
    body    = str(row['body'])    if pd.notna(row['body'])    else "Please review the attached information."

    assistant_reply = f"Subject: {subject}\n\n{body}"

    formatted = f"USER: {COMMON_PROMPT}\nASSISTANT: {assistant_reply}"

    chat_data_strings.append(formatted)

# ───── Create HuggingFace Dataset & split ─────
dataset = Dataset.from_dict({"text": chat_data_strings})
split   = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split["train"]
eval_dataset  = split["test"]

# ───── Save to Drive ─────
train_dataset.save_to_disk(f"{OUTPUT_DIR}/train_dataset")
eval_dataset.save_to_disk(f"{OUTPUT_DIR}/eval_dataset")

print("DONE!")
print(f"Train: {len(train_dataset)} examples")
print(f"Eval : {len(eval_dataset)} examples")
print(f"Saved to → {OUTPUT_DIR}")


print("\n--- EXAMPLE TRAINING ENTRY ---")
print(train_dataset[0]["text"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded and filtered → 1999 safe emails


Saving the dataset (0/1 shards):   0%|          | 0/1799 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

DONE!
Train: 1799 examples
Eval : 200 examples
Saved to → /content/drive/MyDrive/SafeEmailProject

--- EXAMPLE TRAINING ENTRY ---
USER: Generate a realistic legitimate email.
ASSISTANT: Subject: request for information

Dear Colleagues,
The National Terminology Services (NTS) of South Africa is seeking a terminology management system capable of accommodating all 11 official languages. Please note that some African languages have special diacritics not yet available in commercial software.
Attached you will find the RFI from the NTS. The detailed user requirement specification will be emailed to interested parties upon request.
Please take note of the closing date: 29 May. Your assistance in forwarding this information is greatly appreciated.
Yours sincerely,
Ms. Milde Jordan-Weiss National Terminology Services Department of Arts, Culture, Science and Technology Private Bag X8940001 Pretoria Republic of South Africa Tel: +27 12 314-6165 Fax: +27 12 325-4943


In [ ]:
# =========================
# WORKING BLOCK 2: QLoRA TRAINING FOR VICUNA-7B-V1.5
# Uses PyTorch NIGHTLY
# =========================

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install PyTorch NIGHTLY
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu121
# Install latest libraries
!pip install -q --upgrade \
    transformers \
    peft \
    bitsandbytes \
    accelerate \
    trl \
    datasets

import torch
print(f"PyTorch version: {torch.__version__}")
import math
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, TrainerCallback, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_from_disk

# ───── PATHS ─────
OUTPUT_DIR = "/content/drive/MyDrive/SafeEmailProject"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final_model"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# ───── Load datasets ─────
train_dataset = load_from_disk(f"{OUTPUT_DIR}/train_dataset")
eval_dataset  = load_from_disk(f"{OUTPUT_DIR}/eval_dataset")

print(f"Loaded datasets → Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

# ───── Load tokenizer and model (4-bit) ─────
model_name = "arya555/vicuna-7b-v1.5-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# ───── Training arguments ─────
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    weight_decay=0.01,
    bf16=True,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="tensorboard",
    save_total_limit=5,
)

# ───── Perplexity callback ─────
class PerplexityCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        if "eval_loss" in metrics:
            perplexity = math.exp(metrics["eval_loss"])
            print(f"\n=== Step {state.global_step} === Validation Loss: {metrics['eval_loss']:.4f} | Perplexity: {perplexity:.2f}\n")

# ───── SFTTrainer ─────
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    callbacks=[PerplexityCallback()]
)

# ───── TRAIN ─────
print("Starting training... (will resume if checkpoint exists)")
trainer.train(resume_from_checkpoint=False)

# ───── Final evaluation ─────
eval_results = trainer.evaluate()
final_loss = eval_results["eval_loss"]
final_perplexity = math.exp(final_loss)

print(f"\nFINAL RESULTS:")
print(f"Validation Loss: {final_loss:.4f}")
print(f"Validation Perplexity: {final_perplexity:.2f}")

# ───── Save final model ─────
print("\nSaving final LoRA adapter + tokenizer to Drive...")
trainer.model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print(f"Training complete! Model saved to: {FINAL_MODEL_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Looking in indexes: https://download.pytorch.org/whl/nightly/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 18.2 MB/s eta 0:00:00
PyTorch version: 2.9.0+cu126
Loaded datasets → Train: 1799, Eval: 200


tokenizer_config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



=== Step 1200 === Validation Loss: 1.6645 | Perplexity: 5.28



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.652200,1.715839,1.678246,300515.000000,0.618823
400,1.645200,1.685884,1.654386,607007.000000,0.623070
600,1.601400,1.674992,1.657971,902783.000000,0.625398
800,1.541300,1.663539,1.653129,1198048.000000,0.627959
1000,1.469600,1.665539,1.575391,1512133.000000,0.628633
1200,1.429100,1.664498,1.566504,1815538.000000,0.628408



=== Step 1350 === Validation Loss: 1.6635 | Perplexity: 5.28


FINAL RESULTS:
Validation Loss: 1.6635
Validation Perplexity: 5.28

Saving final LoRA adapter + tokenizer to Drive...
Training complete! Model saved to: /content/drive/MyDrive/SafeEmailProject/final_model


In [ ]:
# ==========================================
# 1. INSTALL REQUIRED LIBRARIES
# ==========================================
!pip install -q evaluate bert-score rouge-score transformers peft datasets accelerate

# ==========================================
# 2. IMPORTS
# ==========================================
import math
import torch
import evaluate

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_from_disk

# ==========================================
# 3. PATHS
# ==========================================
BASE_MODEL = "lmsys/vicuna-7b-v1.5"
ADAPTER_PATH = "/content/drive/MyDrive/Vicuna-7B/SafeEmailProject/final_model"
DATASET_PATH = "/content/drive/MyDrive/Vicuna-7B/SafeEmailProject/eval_dataset"

# ==========================================
# 4. LOAD BASE MODEL
# ==========================================
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ==========================================
# 5. LOAD LORA ADAPTER
# ==========================================
model = PeftModel.from_pretrained(
    model,
    ADAPTER_PATH
)

model.eval()

# ==========================================
# 6. LOAD DATASET
# ==========================================
eval_dataset = load_from_disk(DATASET_PATH)

# ==========================================
# 7. TOKENIZE DATASET
# ==========================================
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

eval_dataset = eval_dataset.map(tokenize_function)

eval_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

# ==========================================
# 8. COMPUTE PERPLEXITY
# ==========================================
total_loss = 0.0
sample_count = 0

with torch.no_grad():
    for sample in eval_dataset.select(range(50)):

        input_ids = sample["input_ids"].unsqueeze(0).to(model.device)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(model.device)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        total_loss += outputs.loss.item()
        sample_count += 1

val_loss = total_loss / sample_count
perplexity = math.exp(val_loss)

# ==========================================
# 9. GENERATE PREDICTIONS
# ==========================================
predictions = []
references  = []

for sample in eval_dataset.select(range(30)):

    input_ids = sample["input_ids"].unsqueeze(0).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            max_new_tokens=120,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(output[0], skip_special_tokens=True)

    reference = tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=True
    )

    predictions.append(generated)
    references.append(reference)

# ==========================================
# 10. LOAD METRICS
# ==========================================
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# ==========================================
# 11. FREE GPU MEMORY (IMPORTANT)
# ==========================================
del model
torch.cuda.empty_cache()

# ==========================================
# 12. COMPUTE ROUGE
# ==========================================
rouge_score = rouge.compute(
    predictions=predictions,
    references=references
)

# ==========================================
# 13. COMPUTE BERTSCORE (CPU)
# ==========================================
bert_score = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en",
    model_type="distilbert-base-uncased",
    device="cpu",
    batch_size=4
)

bert_f1 = sum(bert_score["f1"]) / len(bert_score["f1"])
bert_p  = sum(bert_score["precision"]) / len(bert_score["precision"])
bert_r  = sum(bert_score["recall"]) / len(bert_score["recall"])

# ==========================================
# 14. PRINT FINAL RESULTS
# ==========================================
print("\n" + "=" * 50)
print("        EVALUATION RESULTS")
print("=" * 50)

print(f"\nValidation Loss      : {val_loss:.4f}")
print(f"Perplexity           : {perplexity:.4f}")

print(f"\nROUGE-1              : {rouge_score['rouge1']:.4f}")
print(f"ROUGE-2              : {rouge_score['rouge2']:.4f}")
print(f"ROUGE-L              : {rouge_score['rougeL']:.4f}")

print(f"\nBERTScore Precision  : {bert_p:.4f}")
print(f"BERTScore Recall     : {bert_r:.4f}")
print(f"BERTScore F1         : {bert_f1:.4f}")

print("=" * 50)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



        EVALUATION RESULTS

Validation Loss      : 8.7220
Perplexity           : 6136.6168

ROUGE-1              : 0.9386
ROUGE-2              : 0.9374
ROUGE-L              : 0.9378

BERTScore Precision  : 0.9732
BERTScore Recall     : 0.9970
BERTScore F1         : 0.9848


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =========================
# BLOCK 3: GENERATE A SINGLE SAFE EMAIL
# Loads fine-tuned LoRA + base model (safetensors version)
# Uses chat-style prompt matching training format
# =========================

from google.colab import drive
drive.mount('/content/drive')

# Install necessary packages (run once if needed)
!pip install -q --upgrade transformers peft bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

# ───── PATHS  ─────
OUTPUT_DIR = "/content/drive/MyDrive/SafeEmailProject"
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final_model"
BASE_MODEL = "arya555/vicuna-7b-v1.5-hf"

# ───── Load tokenizer ─────
tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ───── 4-bit config for memory efficiency ─────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# ───── Load base model + LoRA adapter ─────
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("Merging LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model,
    FINAL_MODEL_DIR,
    device_map="auto"
)

# ───── Set to evaluation mode ─────
model.eval()

user_prompt = (
    "Generate a realistic legitimate business email from the HR department about the upcoming team-building event next month."
    "Make it urgent and professional."
)

full_prompt = f"USER: {user_prompt}\nASSISTANT:"

# ───── Generate ─────
inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# ───── Extract only the ASSISTANT response ─────
if "ASSISTANT:" in generated_text:
    email_content = generated_text.split("ASSISTANT:", 1)[1].strip()
else:
    email_content = generated_text[len(full_prompt):].strip()

print("\n" + "="*60)
print("GENERATED PHISHING EMAIL:")
print("="*60 + "\n")
print(email_content)
print("\n" + "="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merging LoRA adapter...

GENERATED PHISHING EMAIL:

Subject: Team Building Event - Please RSVP by March 15th

Dear Employees,

Please note that there has been no change in the date for our team building event. The event is scheduled to take place on Saturday, April 27th, starting at 10:00 AM in the Kulshreshtha Hall of Venkatesh International Hotel, Mysore. You are all welcome to participate. It will be an opportunity for you to meet and enjoy each other's company outside of work.

Please RSVP your attendance or non-attendance by March 15th as we need to book accommodation, catering etc. If you have any questions please feel free to contact me directly. Thank you!



In [ ]:
# =========================
# BLOCK 4: GENERATE 2000 SAFE EMAILS & SAVE TO XLSX (WITH PAUSE/RESUME)
# Loads fine-tuned LoRA model
# Generates emails one by one, appending to XLSX
# Tracks progress in a separate file for resume
# If file exists, resumes from current row count
# =========================

from google.colab import drive
drive.mount('/content/drive')

# Install if needed
!pip install -q --upgrade transformers peft bitsandbytes accelerate pandas openpyxl tqdm

import torch
import pandas as pd
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm
import random
# ───── PATHS  ─────
OUTPUT_DIR = "/content/drive/MyDrive/SafeEmailProject"
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final_model"
BASE_MODEL = "arya555/vicuna-7b-v1.5-hf"

GENERATED_XLSX = f"{OUTPUT_DIR}/generated_safe_legitimate_2000.xlsx"
PROGRESS_FILE = f"{OUTPUT_DIR}/safe_generation_progress.txt"

# ───── Config ─────
TOTAL_EMAILS = 2000
BATCH_SIZE = 1
prompt_templates = [
    "Generate a realistic legitimate professional business email.",
    "Create a polite email from HR about the upcoming team performance review.",
    "Write a professional follow-up email after a client meeting.",
    "Generate an internal team email announcing a new project kickoff.",
    "Create a thank-you email to a customer after resolving their support ticket.",
    "Generate a legitimate email from SBI confirming a successful fixed deposit renewal.",
    "Write a polite email from a college professor reminding students about the assignment submission deadline.",
    "Create an email from IRCTC confirming train ticket booking and providing journey details.",
    "Generate a professional email from HDFC Bank about credit card statement availability.",
    "Write an email from a company HR department welcoming a new joiner to the team.",
    "Create a friendly email from PhonePe confirming a successful UPI transaction.",
    "Generate a legitimate email from EPFO about PF passbook update and contribution details.",
    "Write an email thanking a vendor for timely delivery of goods.",
    "Create a professional email scheduling a virtual meeting with agenda points.",
    "Generate an email from a university informing about exam timetable release."
    "Generate a friendly email thanking a friend for their help.",
    "Write a polite email inviting family for a small get-together.",
    "Create an email sharing recent family photos with relatives."
    "Create an email confirming online order shipment with tracking details.",
    "Generate a customer support email resolving a recent query.",
    "Create an email about successful recharge or bill payment.",
    "Write an email announcing a community clean-up drive."
    "Generate an email inviting members to a monthly club meeting."
    "Create an email thanking volunteers for their contribution."
    "Write an email sharing highlights from a recent community event."
    "Generate an email reminding members to renew annual membership."
    "Create an email announcing election results of a student council."
    "Generate an email announcing a workplace wellness session."
    "Write an email inviting employees to a yoga or meditation workshop."
    "Create an email sharing tips for maintaining work-life balance."
    "Generate an email promoting participation in a step-count challenge."
    "Write an email announcing availability of counseling hours."
    "Generate an email confirming refund initiation without asking for bank details."
    "Write an informational email about size exchange availability."
    "Create an email announcing seasonal sale dates (no links)."
    "Generate a customer email sharing tips on product care."
    "Create an informational email reminding users about an upcoming developer event timeline."
    "Write an email announcing shortlisted teams for a coding event."
    "Generate a polite email informing users about mentor session availability."
    "Create an email sharing event rules and code of conduct."
    "Write an email thanking participants for attending a developer workshop."
    "Generate an email announcing prize distribution details after an event."
    "Create an internal update email about changes in event schedule."
]

# ───── Load tokenizer ─────
tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ───── 4-bit config ─────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# ───── Load base + LoRA ─────
print("Loading model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model = PeftModel.from_pretrained(
    base_model,
    FINAL_MODEL_DIR,
    device_map="auto"
)
model.eval()

# ───── Load or init DataFrame ─────
if os.path.exists(GENERATED_XLSX):
    df = pd.read_excel(GENERATED_XLSX)
    start_idx = len(df)
    print(f"Resuming from {start_idx} emails...")
else:
    df = pd.DataFrame(columns=["SUBJECT", "BODY", "LABEL"])
    start_idx = 0

# ───── Generation loop ─────
device = "cuda" if torch.cuda.is_available() else "cpu"

with tqdm(total=TOTAL_EMAILS, initial=start_idx, desc="Generating emails") as pbar:
    for idx in range(start_idx, TOTAL_EMAILS, BATCH_SIZE):
        user_prompt = random.choice(prompt_templates)

        full_prompt = f"USER: {user_prompt}\nASSISTANT:"

        inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=400,
                do_sample=True,
                temperature=0.8,
                top_p=0.95,
                repetition_penalty=1.1,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        if "ASSISTANT:" in generated_text:
            email_content = generated_text.split("ASSISTANT:", 1)[1].strip()
        else:
            email_content = generated_text[len(full_prompt):].strip()

        # Parse Subject and Body
        try:
            parts = email_content.split("\n\n", 1)
            subject = parts[0].replace("Subject: ", "").strip() if parts[0].startswith("Subject: ") else parts[0].strip()
            body = parts[1].strip() if len(parts) > 1 else ""
        except:
            subject = "Important Update"
            body = email_content

        # Append to DF
        new_row = pd.DataFrame({
            "SUBJECT": [subject],
            "BODY": [body],
            "LABEL": [0]
        })
        df = pd.concat([df, new_row], ignore_index=True)

        # Save progress
        df.to_excel(GENERATED_XLSX, index=False)
        with open(PROGRESS_FILE, "w") as f:
            f.write(str(idx + BATCH_SIZE))

        pbar.update(BATCH_SIZE)

print(f"\nGeneration complete! Saved 2000 emails to: {GENERATED_XLSX}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 92.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Loading model...


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Resuming from 1168 emails...


Generating emails: 100%|██████████| 2000/2000 [4:13:35<00:00, 18.29s/it]


Generation complete! Saved 2000 emails to: /content/drive/MyDrive/SafeEmailProject/generated_safe_legitimate_2000.xlsx
